# Scoring recovery
Reconstructed from our conversation, not recovered from the original disk. Keep your original notebook and notes. This companion restores the function and result export; no API calls are made. Warmup means exclude; other sets are assumed to be 0–2 RIR. Coefficients remain 1 / 0.5.

In [2]:
from pathlib import Path
import json
import numpy as np
# Start with the original project; change only this line if needed.
project_folder = Path("C:/Users/AndyM/Projects/hevy")
assets_folder = project_folder / "assets"
data_folder = project_folder / "data"
print("Project:", project_folder)

Project: C:\Users\AndyM\Projects\hevy


In [3]:
with (data_folder / "hevy_workouts.json").open(encoding="utf-8") as f:
    workouts = json.load(f)
with (data_folder / "exercise_templates.json").open(encoding="utf-8") as f:
    templates = json.load(f)
print("Saved workouts:", len(workouts), "Templates:", len(templates))

Saved workouts: 8 Templates: 453


## Reusable scoring function
Rows of A are muscles, columns are exercise entries; the set-count vector multiplies A. All reference muscle labels are retained, including zeros.

In [4]:
def calculate_muscle_scores(workout, templates, primary_weight=1.0, secondary_weight=0.5):
    exercises = workout["exercises"]
    missing = {e["exercise_template_id"] for e in exercises} - set(templates)
    if missing:
        raise ValueError(f"Missing templates: {sorted(missing)}")
    muscle_names = set()
    for template in templates.values():
        if template["primary_muscle_group"]:
            muscle_names.add(template["primary_muscle_group"])
        muscle_names.update(template["secondary_muscle_groups"])
    muscles = sorted(muscle_names)
    muscle_to_row = {m: i for i, m in enumerate(muscles)}
    A = np.zeros((len(muscles), len(exercises)))
    sets = np.zeros(len(exercises))
    for column, exercise in enumerate(exercises):
        template = templates[exercise["exercise_template_id"]]
        primary = template["primary_muscle_group"]
        if primary:
            A[muscle_to_row[primary], column] = primary_weight
        for muscle in template["secondary_muscle_groups"]:
            if muscle != primary:
                A[muscle_to_row[muscle], column] = secondary_weight
        sets[column] = sum(s["type"] != "warmup" for s in exercise["sets"])
    scores = A @ sets
    return {"workout_id": workout["id"], "workout_title": workout["title"],
            "start_time": workout["start_time"],
            "model": {"primary_weight": primary_weight, "secondary_weight": secondary_weight,
                      "excluded_set_type": "warmup"},
            "muscle_scores": dict(zip(muscles, scores.tolist()))}

In [5]:
session = workouts[0]
results = calculate_muscle_scores(session, templates)
print(results["workout_title"], results["start_time"])
print(json.dumps(results["muscle_scores"], indent=2))

I think upper body focus 2026-11-13T21:39:36+00:00
{
  "abdominals": 0.0,
  "abductors": 0.0,
  "adductors": 0.0,
  "biceps": 8.0,
  "calves": 3.5,
  "cardio": 1.0,
  "chest": 4.0,
  "forearms": 6.0,
  "full_body": 0.0,
  "glutes": 2.0,
  "hamstrings": 9.0,
  "lats": 6.0,
  "lower_back": 0.0,
  "neck": 0.0,
  "other": 0.0,
  "quadriceps": 4.0,
  "shoulders": 2.0,
  "traps": 0.0,
  "triceps": 2.0,
  "upper_back": 6.0
}


## Save a recovery result without replacing the existing result

In [6]:
output_path = data_folder / "latest_workout_scores.json"

with output_path.open("w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

print("Saved:", output_path)

Saved: C:\Users\AndyM\Projects\hevy\data\latest_workout_scores.json
